# 🤖 SentinelX AI - AMDO / Jupyter Launchpad

Welcome to the SentinelX AI cloud environment. This notebook is designed to run the entire FastAPI + AI Core platform directly within a Jupyter cell.

**Why this works better here:**
- **Linux-based:** AMDO runs on Linux, which completely bypasses the `WinError 10054` asyncio bug you were seeing on Windows.
- **GPU Accelerated:** It can leverage the ROCm/vLLM environment if available.
- **All-in-one:** `nest_asyncio` allows the heavy Uvicorn server to run right here in the notebook.

### Instructions:
1. Ensure all project files (`api/`, `logic/`, `web_ui/`, etc.) are uploaded to the same directory as this notebook.
2. Run **Cell 1** to install dependencies.
3. Run **Cell 2** to launch the platform.
4. Click the generated `https://...` link in the output to access the dashboard.

In [ ]:
# CELL 1: Environment Setup
!pip install -r requirements.txt

import os
import requests
from tqdm.notebook import tqdm

# Download LLM Model if missing
model_dir = "models"
model_name = "tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf"
model_path = os.path.join(model_dir, model_name)
url = "https://huggingface.co/TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF/resolve/main/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf"

if not os.path.exists(model_dir):
    os.makedirs(model_dir)

if not os.path.exists(model_path):
    print(f"Downloading LLM Model: {model_name}...")
    response = requests.get(url, stream=True)
    total_size = int(response.headers.get('content-length', 0))
    with open(model_path, 'wb') as f, tqdm(
        desc=model_name,
        total=total_size,
        unit='iB',
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for data in response.iter_content(chunk_size=1024):
            size = f.write(data)
            bar.update(size)
    print("Model downloaded.")
else:
    print("Model already exists.")

In [ ]:
# CELL 2: Launch Platform
import nest_asyncio
import uvicorn
import generate_cert
import logging
from logic.system_core import SystemCore
from api.server import app, set_core

# 1. Patch asyncio to allow Uvicorn to run inside Jupyter
nest_asyncio.apply()

# 2. Generate SSL (Required for Browser Microphone Access)
generate_cert.generate_self_signed_cert()

# 3. Initialize Core AI Threads
print("Initializing SentinelX Core...")
core = SystemCore()
core.start()
set_core(core)

# 4. Start Server
print("\n" + "="*50)
print("🚀 SYSTEM READY 🚀")
print("Access the UI at: https://localhost:8000")
print("Note: You may need to click 'Advanced' -> 'Proceed' in your browser due to the local SSL cert.")
print("="*50 + "\n")

uvicorn.run(
    app, 
    host="0.0.0.0", 
    port=8000, 
    ssl_keyfile="key.pem", 
    ssl_certfile="cert.pem",
    log_level="warning" # Suppress normal traffic logs to keep notebook clean
)